# Quality Control

## Load count matrix

In [1]:
import numpy as np
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation

sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

In [2]:
import tempfile
from pathlib import Path

import anndata
import anndata2ri
from rpy2.robjects import r
from scipy.sparse import csr_matrix

anndata2ri.activate()
%load_ext rpy2.ipython

/tmp/ipykernel_27107/1601398045.py:9: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [4]:
import rpy2
%load_ext rpy2.ipython

In [32]:
%%R -o sce
# Read the combined, empty-drop filtered count matrix RDS file and output SingleCellExperiment object to Python
rds_path <- "../raw_data_processing/results/alevin/mtx_conversions/combined_emptydrops_filter_matrix.sce.rds"
sce <- readRDS(rds_path)

# Extract Ensembl gene IDs
ensembl_ids <- rownames(sce)

# Convert Ensembl IDs to gene symbols
gene_symbols <- mapIds(EnsDb.Hsapiens.v86, 
                       keys = ensembl_ids, 
                       column = "SYMBOL", 
                       keytype = "GENEID", 
                       multiVals = "first")

# Convert Ensembl IDs to full gene names
gene_names <- mapIds(EnsDb.Hsapiens.v86, 
                     keys = ensembl_ids, 
                     column = "GENENAME", 
                     keytype = "GENEID", 
                     multiVals = "first")

# Add gene symbols and gene names to rowData
rowData(sce)$gene_symbol <- gene_symbols
rowData(sce)$gene_name <- gene_names

# Check the first few mappings
head(rowData(sce))



DataFrame with 6 rows and 3 columns
                  gene_versions gene_symbol   gene_name
                    <character> <character> <character>
ENSG00000001461 ENSG00000001461      NIPAL3      NIPAL3
ENSG00000010072 ENSG00000010072       SPRTN       SPRTN
ENSG00000008118 ENSG00000008118      CAMK1G      CAMK1G
ENSG00000009780 ENSG00000009780      FAM76A      FAM76A
ENSG00000048707 ENSG00000048707      VPS13D      VPS13D
ENSG00000041988 ENSG00000041988       THAP3       THAP3


In [33]:
print(sce)

class: SingleCellExperiment 
dim: 236796 1189 
metadata(0):
assays(1): X
rownames(236796): ENSG00000001461 ENSG00000010072 ... ENSG00000310554-A
  ENSG00000310555-A
rowData names(3): gene_versions gene_symbol gene_name
colnames(1189): GCATGTACAATCTACG_Hair_graft_5_emptydrops_filter
  TGACAACAGATGTGGC_Hair_graft_5_emptydrops_filter ...
  CGATTGATCTGTGCAA_Hair_graft_5_emptydrops_filter
  ACCTTTAGTTTAGGAA_Hair_graft_5_emptydrops_filter
colData names(3): sample fastq_1 fastq_2
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):



In [34]:
# Convert the SingleCellExperiment object to AnnData object
import anndata2ri
from rpy2.robjects import r
from rpy2.robjects.conversion import localconverter

with localconverter(anndata2ri.converter):
    adata = r('as(sce, "SingleCellExperiment")')

print(adata)

AnnData object with n_obs × n_vars = 1189 × 236796
    obs: 'sample', 'fastq_1', 'fastq_2'
    var: 'gene_versions', 'gene_symbol', 'gene_name'


In [35]:
import pandas as pd
gene_df = pd.DataFrame(adata.var)
gene_df

,gene_versions,gene_symbol,gene_name
ENSG00000001461,ENSG00000001461,NIPAL3,NIPAL3
ENSG00000010072,ENSG00000010072,SPRTN,SPRTN
ENSG00000008118,ENSG00000008118,CAMK1G,CAMK1G
ENSG00000009780,ENSG00000009780,FAM76A,FAM76A
ENSG00000048707,ENSG00000048707,VPS13D,VPS13D
...,...,...,...
ENSG00000310550-A,ENSG00000310550-A,None,None
ENSG00000310552-A,ENSG00000310552-A,None,None
ENSG00000310553-A,ENSG00000310553-A,None,None
ENSG00000310554-A,ENSG00000310554-A,None,None
